In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [5]:
start = dt.datetime(2019,5,15)
start = start.replace(hour=0, minute=0, second=0, microsecond=0)
#start = start - dt.timedelta(hours = 5, minutes = 30)
end = dt.datetime(2019,5,20)
end = end.replace(hour=0, minute=0, second=0, microsecond=0)
#end = end - dt.timedelta(hours = 5, minutes = 30)
#print(start,end,end-start)

In [7]:
base = '1970-01-01'
x = (start - pd.to_datetime(base)).total_seconds()
y = (end - pd.to_datetime(base)).total_seconds()

x = int(x-(5.5*3600))
y = int(y-(5.5*3600))
print(x)
print(y)

1557858600
1558290600


In [8]:
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
query = (f"""SELECT
            device_id,
            (event_timestamp) as event_timestamp 
        FROM
            `hitwicketsuperstars.sessions.sessions_*` 
            WHERE (event_timestamp >= {x}*1000000 AND event_timestamp <= {y+86400}*1000000)"""
        )
df = client.query(query).to_dataframe()
df.head()

,device_id,event_timestamp
0,af1a793ea82064b5870c6a0b6c529bd2,1558005571912000
1,c800950c4a5b84c88e1f328a5e6fa68d,1558030288918000
2,33d2a864ee7d1804c893156d148fb6be,1557996171122000
3,312a918e697d414d1f6e11012b628c89,1558010400230000
4,fd71261cfa2c659c6c7e0880839c9907,1557980441875000


In [9]:
orig_query_df = df.copy()

In [10]:
df = orig_query_df.copy()

In [11]:
len(df)

25039

In [13]:
df.sort_values('event_timestamp',inplace= True)
df.head()

,device_id,event_timestamp
7048,b0041f4027448da9fe1047e3c9ac1df5,1557858612283000
5190,0dfe47e6ca041992e4b8d9dcc56a1d0c,1557858654071000
7364,4f55bca78de5bc301f12fb4d32302a9d,1557858679372000
8409,4f3f93aeab324846c05747c9b67dbe23,1557858701543000
10338,20a89d8fa5881bb118865675792f0fec,1557858701882000


In [14]:
df['event_timestamp'] = pd.to_datetime(df['event_timestamp'],unit = 'us')
df['event_timestamp'] = df['event_timestamp'] + dt.timedelta(hours = 5, minutes = 30)

In [15]:
df['event_timestamp'] = df['event_timestamp'].dt.date

In [16]:
df.head()

,device_id,event_timestamp
7048,b0041f4027448da9fe1047e3c9ac1df5,2019-05-15
5190,0dfe47e6ca041992e4b8d9dcc56a1d0c,2019-05-15
7364,4f55bca78de5bc301f12fb4d32302a9d,2019-05-15
8409,4f3f93aeab324846c05747c9b67dbe23,2019-05-15
10338,20a89d8fa5881bb118865675792f0fec,2019-05-15


In [17]:
df.columns = ['device_id','time']
df.head()

,device_id,time
7048,b0041f4027448da9fe1047e3c9ac1df5,2019-05-15
5190,0dfe47e6ca041992e4b8d9dcc56a1d0c,2019-05-15
7364,4f55bca78de5bc301f12fb4d32302a9d,2019-05-15
8409,4f3f93aeab324846c05747c9b67dbe23,2019-05-15
10338,20a89d8fa5881bb118865675792f0fec,2019-05-15


In [21]:
len(df)

25039

In [22]:
ab = df.sort_values(['device_id','time'])
ab.drop_duplicates(['device_id','time'],inplace = True)
print(len(ab))
ab.head()

16284


,device_id,time
600,0005bf93180bdd4b361c2e115dbee2d7,2019-05-16
9705,0007a83f7338f75d905a007245b1b485,2019-05-15
3960,0007a83f7338f75d905a007245b1b485,2019-05-16
14363,0007a83f7338f75d905a007245b1b485,2019-05-17
18198,0007a83f7338f75d905a007245b1b485,2019-05-18


In [23]:
df['time'].astype('str').unique().tolist()

['2019-05-15',
 '2019-05-16',
 '2019-05-17',
 '2019-05-18',
 '2019-05-19',
 '2019-05-20']

In [24]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
c_users = cursor.superstars.users
aw = []
for documents in c_users.find({},{"sign_up_details":1, "created_at":1}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df1 = pd.DataFrame(dic_flattened)
users = df1[["_id","created_at","sign_up_details_device_id"]]
users.columns = ["user_id","create_time","device_id"]
print(len(users))
users.head()

128762


,user_id,create_time,device_id
0,5c702c13453bed5fa3bdba59,2019-02-22 17:06:27.769,0dbb1458df0b26c924a23c4419f79506
1,5c702f100899f358b8f31db7,2019-02-22 17:19:12.954,00da33af8422e56009693e6351ce39a1
2,5c702f1a0899f358b8f31dd6,2019-02-22 17:19:22.914,11476de18d2f222f3c18b846018d7c2b
3,5c702f9865a1f762ddc8cbe6,2019-02-22 17:21:28.152,a8fde2641d175a8c7274416f69fca69f
4,5c70308f0899f358b8f31e19,2019-02-22 17:25:35.387,3f6e334678472989053110fa7c5f2c4e


In [34]:
rf = users.drop_duplicates('device_id')
len(rf)

125178

In [35]:
df2 = pd.merge(ab,rf,on='device_id',how = 'inner')
print(len(df2))
df2.head()

10028


,device_id,time,user_id,create_time
0,0005bf93180bdd4b361c2e115dbee2d7,2019-05-16,5cd845b1af1b883aa3714302,2019-05-12 16:11:29.333
1,0007a83f7338f75d905a007245b1b485,2019-05-15,5ca894b4ef46c6361e50788e,2019-04-06 11:59:48.641
2,0007a83f7338f75d905a007245b1b485,2019-05-16,5ca894b4ef46c6361e50788e,2019-04-06 11:59:48.641
3,0007a83f7338f75d905a007245b1b485,2019-05-17,5ca894b4ef46c6361e50788e,2019-04-06 11:59:48.641
4,0007a83f7338f75d905a007245b1b485,2019-05-18,5ca894b4ef46c6361e50788e,2019-04-06 11:59:48.641


In [36]:
grouped_df = df2.groupby('time')

In [37]:
dic={}
for i in grouped_df.groups:
    dic[str(i)] = set(grouped_df.get_group(i)['user_id'].tolist())

In [39]:
df3 = pd.DataFrame([dic],index=['Users'])
df3

,2019-05-15,2019-05-16,2019-05-17,2019-05-18,2019-05-19,2019-05-20
Users,"{5cda45d24552a85306dfdb02, 5ca7a30f2f276d26a69...","{5cda45d24552a85306dfdb02, 5caf5b289adc5545e4e...","{5cd2b29b5f935514f8dee745, 5cd662ad0e9c8d17cfd...","{5cda45d24552a85306dfdb02, 5ca7a30f2f276d26a69...","{5ca7a30f2f276d26a69b5385, 5cd131f9fe850e7a1ce...","{5ca7a30f2f276d26a69b5385, 5cd787bee839226f40a..."


In [40]:
total=0
for i in df3.iloc[0]:
    print(len(list(i)))
    total+=len(list(i))

2323
1969
1730
1514
1379
1113


In [41]:
total

10028

In [58]:
c_pay = cursor.superstars.payment_orders
aw = []
for documents in c_pay.find({'created_at': {'$lt': end, '$gte': start}}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df4 = pd.DataFrame(dic_flattened)
payment = df4[["_id","user_id","status","revenue_bottom_line","revenue_top_line"]]
payment = payment[payment['status']==2]
payment.columns = ["pay_id","user_id","status","revenue_bottom_line","revenue_top_line"]
print(len(payment))
payment.head()

2


,pay_id,user_id,status,revenue_bottom_line,revenue_top_line
1,5ce009497cba091cb055b5c0,5cc1bb1c559c2d25276ead10,2,0.0,0.0
2,5ce102e87cba091cb05bd3bc,5cd7d1170719fb6f18b8952b,2,139.3,199.0


In [54]:
total_top_revenue = payment['revenue_top_line'].sum()
total_bot_revenue = payment['revenue_bottom_line'].sum()

In [56]:
arpu = total_bot_revenue/total

In [57]:
arpu

0.013891104906262464